# Causal Inference - Beyond A/B Testing

## Introduction

Welcome to the **ADVANCED PHASE**: Causal Inference in A/B Testing.

### The Problem with Traditional A/B Testing

The previous analysis showed us:
- **P-value = 0.19** -> Not significant
- **Uplift = -0.16%** -> Negative
- **Recommendation**: DO NOT LAUNCH

But there are deeper questions:

> - Does the TREATMENT actually CAUSE the difference?
> - Or are there confounding variables that explain the relationship?
> - Is the effect the same for EVERYONE or does it vary by subgroup?
> - How much CONFIDENCE do we have in the causal effect?

### Objective of This Notebook

Use **Causal Inference** to:
1. Estimate the CAUSAL effect of the treatment
2. Control for confounding variables
3. Estimate heterogeneous effects (CATE)
4. Make an informed and robust decision

## STEP 1: Key Concepts - Causal Inference

### 1. Correlation vs Causality

**Correlation** = two variables vary together
- Example: Ice cream <-> Crime (both increase in summer)
- Relationship: real but SPURIOUS

**Causality** = change in A CAUSES change in B
- Example: Clicking "Buy" -> Purchase completed
- Relationship: X->Y (clear directionality)

### 2. Confounding Variables

**Confounding Variable** = affects both the treatment and the outcome

```
Confounder
   |
   v
Treatment -----> Outcome
```

Example in an A/B test:
- Variable: Time of day
- Nighttime users -> See treatment
- Nighttime users -> Lower conversion (naturally)
- OBSERVED effect: Treatment causes a drop
- REAL effect: Night causes the drop, not treatment

### 3. Average Treatment Effect (ATE)

$$\text{ATE} = E[Y_i(1)] - E[Y_i(0)]$$

- $Y_i(1)$ = outcome if user i receives treatment
- $Y_i(0)$ = outcome if user i does NOT receive treatment
- ATE = Average treatment effect across the ENTIRE population

**Interpreting ATE:**
- ATE = 0.02 -> Treatment increases conversion by 2 points
- ATE = -0.01 -> Treatment decreases conversion by 1 point
- ATE = 0 -> Treatment has no effect

### 4. Conditional Average Treatment Effect (CATE)

$$\text{CATE}(x) = E[Y_i(1) - Y_i(0) | X_i = x]$$

- CATE = Treatment effect CONDITIONED on characteristics X
- Allows detection of **effect heterogeneity**

**Example:**
- CATE(day) = +0.05 -> During the day, treatment improves by 5%
- CATE(night) = -0.02 -> At night, treatment worsens by 2%
- Effect VARIES by time of day

### 5. Assumptions for Causality

To estimate a causal ATE, we need:

**1. Unconfoundedness (Ignorability)**
- No unobserved confounding variables
- Given X, treatment assignment is random
- In RCT: Guaranteed by randomization

**2. Overlap (Common Support)**
- Both groups have similar probability of treatment
- We do not want 100% of a population in treatment

**3. No Interference (SUTVA)**
- Treatment of user A does not affect the outcome of user B
- On digital platforms: Sometimes violated

## STEP 2: Prepare Dataset

### Required Variables

- `treatment`: 0 (old_page) / 1 (new_page) 
- `conversion`: 0 / 1 (outcome)
- `timestamp`: When the user visited
- `hour`: Derived variable (for CATE analysis)
- `time_segment`: day/night (for CATE analysis)

### Load clean data from the previous notebook

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load Udacity dataset (already cleaned from the previous notebook)
path = '../data/ab_data.csv'
df = pd.read_csv(path)

# Clean (remove inconsistencies)
df = df[
    ((df["group"] == "treatment") & (df["landing_page"] == "new_page")) |
    ((df["group"] == "control") & (df["landing_page"] == "old_page"))
]

# Create key variables
df["treatment"] = (df["group"] == "treatment").astype(int)
df["conversion"] = df["converted"].astype(int)

# Parse timestamp
df["timestamp"] = pd.to_datetime(df["timestamp"])
df["hour"] = df["timestamp"].dt.hour

# Create time segments
def categorize_time(hour):
    if 8 <= hour <= 20:
        return "day"
    else:
        return "night"

df["time_segment"] = df["hour"].apply(categorize_time)

print(f"Dataset loaded: {len(df):,} users")
print()
print("Key variables:")
print(df[["treatment", "conversion", "hour", "time_segment"]].head(10))
print()
print("Statistics:")
print(f"  Treatment: {df['treatment'].mean():.1%}")
print(f"  Conversion: {df['conversion'].mean():.2%}")
print(f"  Average hour: {df['hour'].mean():.1f}")

Dataset loaded: 290,585 users

Key variables:
   treatment  conversion  hour time_segment
0          0           0    22        night
1          0           0     8          day
2          1           0    16          day
3          1           0    18          day
4          0           1     1        night
5          0           0    15          day
6          1           1     3        night
7          0           0     1        night
8          1           1    17          day
9          1           1    18          day

Statistics:
  Treatment: 50.0%
  Conversion: 11.96%
  Average hour: 11.5


## STEP 3: Estimate Simple ATE

### Methodology

The simplest ATE is:

$$\text{ATE} = \bar{Y}_{\text{treatment}} - \bar{Y}_{\text{control}}$$

That is, the **difference in means**.

### Interpretation

- If ATE > 0: Treatment IMPROVES the outcome
- If ATE < 0: Treatment WORSENS the outcome
- If ATE = 0: No effect

### Limitation

This ATE does NOT control for confounding variables.
If there is confounding -> ATE is biased.

In [3]:
print("="*70)
print("STEP 3: ESTIMATE SIMPLE ATE")
print("="*70)
print()

# Calculate conversion by group
control_outcome = df[df["treatment"] == 0]["conversion"]
treatment_outcome = df[df["treatment"] == 1]["conversion"]

control_mean = control_outcome.mean()
treatment_mean = treatment_outcome.mean()

# Estimated ATE
ate_simple = treatment_mean - control_mean
ate_simple_pct = ate_simple * 100

print(f"CONTROL OUTCOME (mean):")
print(f"  E[Y(0)] = {control_mean:.4f} ({control_mean:.2%})")
print()
print(f"TREATMENT OUTCOME (mean):")
print(f"  E[Y(1)] = {treatment_mean:.4f} ({treatment_mean:.2%})")
print()
print(f"AVERAGE TREATMENT EFFECT (ATE):")
print(f"  ATE = E[Y(1)] - E[Y(0)]")
print(f"  ATE = {treatment_mean:.4f} - {control_mean:.4f}")
print(f"  ATE = {ate_simple:.4f} ({ate_simple_pct:+.2f}% points)")
print()

# Interpretation
if ate_simple > 0:
    print(f"Interpretation: Treatment IMPROVES conversion by {abs(ate_simple_pct):.2f}% points")
else:
    print(f"Interpretation: Treatment WORSENS conversion by {abs(ate_simple_pct):.2f}% points")

print()
print("NOTE: This ATE does NOT control for confounders")
print("   May be biased if there are omitted confounding variables")
print("="*70)

STEP 3: ESTIMATE SIMPLE ATE

CONTROL OUTCOME (mean):
  E[Y(0)] = 0.1204 (12.04%)

TREATMENT OUTCOME (mean):
  E[Y(1)] = 0.1188 (11.88%)

AVERAGE TREATMENT EFFECT (ATE):
  ATE = E[Y(1)] - E[Y(0)]
  ATE = 0.1188 - 0.1204
  ATE = -0.0016 (-0.16% points)

Interpretation: Treatment WORSENS conversion by 0.16% points

NOTE: This ATE does NOT control for confounders
   May be biased if there are omitted confounding variables


## STEP 4: OLS Regression - ATE with Inference

### Why Regression?

Difference in means gives us a point ATE.
But it does not give us **uncertainty** (standard error, p-value, CI).

OLS regression allows us to:
1. Estimate ATE
2. Calculate standard error
3. Significance test
4. Confidence intervals

### Simple Model

$$Y_i = \beta_0 + \beta_1 \cdot \text{Treatment}_i + \epsilon_i$$

- $\beta_1$ = ATE (the causal effect)
- $\beta_0$ = mean outcome for control
- $\epsilon_i$ = error (unexplained variation)

### Interpreting Results

- **Treatment Coefficient**: Estimated causal effect
- **P-value < 0.05**: Significant
- **CI (Confidence Interval)**: Likely range of the effect

In [4]:
print("="*70)
print("STEP 4: OLS REGRESSION - ATE WITH INFERENCE")
print("="*70)
print()

# Simple model: Y = b0 + b1*Treatment + e
X = df[["treatment"]]
X = sm.add_constant(X)  # Adds const (b0)
y = df["conversion"]

model_simple = sm.OLS(y, X).fit()

print("MODEL: Conversion = b0 + b1*Treatment")
print()
print(model_simple.summary())
print()

# Extract key results
ate_ols = model_simple.params["treatment"]
ate_se = model_simple.bse["treatment"]
ate_pval = model_simple.pvalues["treatment"]
ate_ci_lower = model_simple.conf_int().loc["treatment", 0]
ate_ci_upper = model_simple.conf_int().loc["treatment", 1]

print("="*70)
print("KEY RESULTS:")
print("="*70)
print()
print(f"ATE (treatment coef.):  {ate_ols:.6f}")
print(f"Std Error:              {ate_se:.6f}")
print(f"P-value:                {ate_pval:.6f}")
print(f"95% CI:                 [{ate_ci_lower:.6f}, {ate_ci_upper:.6f}]")
print()

# Interpretation
print("INTERPRETATION:")
print(f"  - Treatment causes a change of {ate_ols:.4f} ({ate_ols*100:+.2f}% points)")
if ate_pval < 0.05:
    print(f"  - This effect IS SIGNIFICANT (p={ate_pval:.4f} < 0.05)")
else:
    print(f"  - This effect is NOT significant (p={ate_pval:.4f} > 0.05)")
print(f"  - 95% probability that real effect is in [{ate_ci_lower:.4f}, {ate_ci_upper:.4f}]")
print()
print("="*70)

STEP 4: OLS REGRESSION - ATE WITH INFERENCE

MODEL: Conversion = b0 + b1*Treatment

                            OLS Regression Results                            
Dep. Variable:             conversion   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                  0.000
Method:                 Least Squares   F-statistic:                     1.720
Date:                Thu, 09 Apr 2026   Prob (F-statistic):              0.190
Time:                        11:42:04   Log-Likelihood:                -85267.
No. Observations:              290585   AIC:                         1.705e+05
Df Residuals:                  290583   BIC:                         1.706e+05
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------

## STEP 5: Add Covariates - Control for Bias

### Confounding in Our Data

Recall from the previous analysis:
- DAYTIME: Treatment +0.03% (neutral)
- NIGHTTIME: Treatment -0.38% (negative)

**Potential confounder**: `hour` (time of day)
- If treatment was assigned more at night
- And nighttime naturally has lower conversion
- Then treatment appears worse than it really is

### Solution: Include Covariates

$$Y_i = \beta_0 + \beta_1 \cdot \text{Treatment}_i + \beta_2 \cdot \text{Hour}_i + \epsilon_i$$

**New b1** = Effect of Treatment CONTROLLING for Hour
- We compare treatment vs control AT THE SAME HOUR
- We remove bias caused by hour

### Control Strategy

1. Include variables that:
   - Affect the outcome (conversion)
   - Vary between groups (correlated with treatment)

2. Do NOT include:
   - Variables that are RESULTS of the treatment (mediators)
   - Highly collinear variables

In [5]:
print("="*70)
print("STEP 5: ADD COVARIATES - CONTROL FOR BIAS")
print("="*70)
print()

# Create control variables
# 1. Nighttime indicator
df["night"] = (df["time_segment"] == "night").astype(int)

# 2. Hour as continuous variable (normalized)
df["hour_normalized"] = (df["hour"] - df["hour"].mean()) / df["hour"].std()

print("INCLUDED COVARIATES:")
print("  1. hour_normalized: Time of day (normalized)")
print("  2. night: Indicator for nighttime (0/1)")
print()

# Model WITH covariates
X_with_covars = df[["treatment", "hour_normalized", "night"]]
X_with_covars = sm.add_constant(X_with_covars)
y = df["conversion"]

model_with_covars = sm.OLS(y, X_with_covars).fit()

print("MODEL: Conversion = b0 + b1*Treatment + b2*Hour_normalized + b3*Night")
print()
print(model_with_covars.summary())
print()

# Compare results
ate_simple = model_simple.params["treatment"]
ate_with_covars = model_with_covars.params["treatment"]
ate_simple_ci = model_simple.conf_int().loc["treatment", :]
ate_covars_ci = model_with_covars.conf_int().loc["treatment", :]

print("="*70)
print("COMPARISON: Simple Model vs With Covariates")
print("="*70)
print()
print(f"WITHOUT COVARIATES (potential bias):")
print(f"  ATE = {ate_simple:.6f} [{ate_simple_ci[0]:.6f}, {ate_simple_ci[1]:.6f}]")
print()
print(f"WITH COVARIATES (bias controlled):")
print(f"  ATE = {ate_with_covars:.6f} [{ate_covars_ci[0]:.6f}, {ate_covars_ci[1]:.6f}]")
print()
print(f"DIFFERENCE: {ate_simple - ate_with_covars:.6f}")
print()
print("INTERPRETATION:")
print(f"  - Without covariates: ATE = {ate_simple*100:.2f}% (BIASED)")
print(f"  - With covariates: ATE = {ate_with_covars*100:.2f}% (CONTROLLED)")
if abs(ate_simple - ate_with_covars) > 0.0005:
    print(f"  - The bias was substantial ({(ate_simple - ate_with_covars)*100:.2f}% points)")
    print(f"  - Including covariates MATTERS")
else:
    print(f"  - The bias was small")
print()
print("="*70)

STEP 5: ADD COVARIATES - CONTROL FOR BIAS

INCLUDED COVARIATES:
  1. hour_normalized: Time of day (normalized)
  2. night: Indicator for nighttime (0/1)

MODEL: Conversion = b0 + b1*Treatment + b2*Hour_normalized + b3*Night

                            OLS Regression Results                            
Dep. Variable:             conversion   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                  0.000
Method:                 Least Squares   F-statistic:                     3.335
Date:                Thu, 09 Apr 2026   Prob (F-statistic):             0.0185
Time:                        11:42:04   Log-Likelihood:                -85263.
No. Observations:              290585   AIC:                         1.705e+05
Df Residuals:                  290581   BIC:                         1.706e+05
Df Model:                           3                                         
Covariance Type:            nonrobust                           

## STEP 6: CATE - Conditional Average Treatment Effect

### Why Estimate CATE?

ATE is the **AVERAGE** effect.
But the effect can **vary** across subgroups.

### Example in Our Data

We know:
- DAYTIME: Treatment nearly neutral (+0.03%)
- NIGHTTIME: Treatment negative (-0.38%)

**Average**: -0.16% (simple ATE)
**But**: Effect is HETEROGENEOUS (different by hour)

### Methodology: Regression with Interaction

$$Y_i = \beta_0 + \beta_1 \cdot T_i + \beta_2 \cdot X_i + \beta_3 \cdot (T_i \times X_i) + \epsilon_i$$

- $\beta_3$ = interaction = how Treatment x Time_Segment interacts
- Allows the effect of T to vary with X

### Interpreting Interactions

If $\beta_3 \neq 0$ -> Heterogeneous effect
If $\beta_3 = 0$ -> Homogeneous effect (same for everyone)

In [6]:
print("="*70)
print("STEP 6: CATE - CONDITIONAL AVERAGE TREATMENT EFFECT")
print("="*70)
print()

# Create dummy variables for time_segment
df["day_segment"] = (df["time_segment"] == "day").astype(int)

# Create interaction: treatment x time_segment
df["treatment_x_day"] = df["treatment"] * df["day_segment"]

# Model WITH interaction
X_interaction = df[["treatment", "day_segment", "treatment_x_day"]]
X_interaction = sm.add_constant(X_interaction)
y = df["conversion"]

model_interaction = sm.OLS(y, X_interaction).fit()

print("MODEL: Conversion = b0 + b1*Treatment + b2*Day + b3*(Treatment x Day)")
print()
print(model_interaction.summary())
print()

# Extract conditional effects
beta_treatment = model_interaction.params["treatment"]
beta_interaction = model_interaction.params["treatment_x_day"]

# CATE by segment
cate_night = beta_treatment  # Effect when day_segment = 0
cate_day = beta_treatment + beta_interaction  # Effect when day_segment = 1

# CI for each CATE
cate_night_se = model_interaction.bse["treatment"]
cate_day_se = np.sqrt(model_interaction.bse["treatment"]**2 + model_interaction.bse["treatment_x_day"]**2)  # Approximation

print("="*70)
print("CONDITIONAL AVERAGE TREATMENT EFFECTS (CATE)")
print("="*70)
print()
print(f"CATE for NIGHT (8pm-8am):")
print(f"  Effect = {cate_night:.6f} ({cate_night*100:+.2f}% points)")
print()
print(f"CATE for DAY (8am-8pm):")
print(f"  Effect = {cate_day:.6f} ({cate_day*100:+.2f}% points)")
print()
print(f"HETEROGENEITY (difference between groups):")
print(f"  b3 (interaction) = {beta_interaction:.6f}")
print(f"  P-value = {model_interaction.pvalues['treatment_x_day']:.6f}")
if model_interaction.pvalues['treatment_x_day'] < 0.05:
    print(f"  SIGNIFICANT HETEROGENEITY: Effect VARIES by time of day")
else:
    print(f"  Heterogeneity is NOT significant: Effect is similar across all segments")
print()
print("="*70)
print()

# Visualize CATE
fig, ax = plt.subplots(figsize=(10, 6))
segments = ['Night\n(8pm-8am)', 'Day\n(8am-8pm)']
cates = [cate_night, cate_day]
ses = [cate_night_se, cate_day_se]
colors = ['#FF6B6B', '#4ECDC4']

bars = ax.bar(segments, np.array(cates)*100, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
ax.errorbar(segments, np.array(cates)*100, yerr=np.array(ses)*100*1.96, 
            fmt='none', ecolor='black', capsize=5, capthick=2, linewidth=2)

ax.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5)
ax.set_ylabel('CATE (% points)', fontweight='bold', fontsize=12)
ax.set_title('Heterogeneity: Treatment Effect by Time Segment', fontweight='bold', fontsize=14)
ax.grid(axis='y', alpha=0.3)

# Add values on bars
for bar, cate in zip(bars, np.array(cates)*100):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
           f'{cate:+.2f}%', ha='center', va='bottom' if height > 0 else 'top', fontweight='bold')

plt.tight_layout()
plt.show()

print("CATE chart generated")

Dataset loaded: 290,585 users

Key variables:
   treatment  conversion  hour time_segment
0          0           0    22        night
1          0           0     8          day
2          1           0    16          day
3          1           0    18          day
4          0           1     1        night
5          0           0    15          day
6          1           1     3        night
7          0           0     1        night
8          1           1    17          day
9          1           1    18          day

Statistics:
  Treatment: 50.0%
  Conversion: 11.96%
  Average hour: 11.5


## STEP 7: Deep Interpretation - Is it Causal?

### Central Question

> **Is the effect we estimated truly CAUSAL?**

Recall: For causality we need:
1. **Unconfoundedness** (verified in RCT)
2. **Overlap** (both groups have users at all hours)
3. **No Interference** (probabilistic at user scale)

### Validate Assumptions

### Interpret Heterogeneity

### Compare with Previous Analysis

### Sources of Uncertainty

### Robustness Checks

In [7]:
print("="*70)
print("STEP 7: DEEP INTERPRETATION - IS IT CAUSAL?")
print("="*70)
print()

print("1. IS IT REALLY CAUSAL?")
print("-" * 70)
print()
print("Validate causality assumptions:")
print()
print("  a) UNCONFOUNDEDNESS (Randomization)")
mean_control_hour = df[df['treatment']==0]['hour'].mean()
mean_treatment_hour = df[df['treatment']==1]['hour'].mean()
hour_diff = abs(mean_control_hour - mean_treatment_hour)
if hour_diff < 0.1:
    print("     Treatment vs Control have SIMILAR hour distribution")
    print(f"       - Mean hour control: {mean_control_hour:.2f}")
    print(f"       - Mean hour treatment: {mean_treatment_hour:.2f}")
    print("     -> No selective confounding by hour")
else:
    print("     Distributions differ by hour")
print()
print("  b) OVERLAP (Common Support)")
overlap_control = (df[df['treatment']==0]['hour'].min(), df[df['treatment']==0]['hour'].max())
overlap_treatment = (df[df['treatment']==1]['hour'].min(), df[df['treatment']==1]['hour'].max())
print(f"     Hours in control: {overlap_control}")
print(f"     Hours in treatment: {overlap_treatment}")
if overlap_control[0] == overlap_treatment[0] and overlap_control[1] == overlap_treatment[1]:
    print("     Full OVERLAP: both groups see all hours")
else:
    print("     Partial overlap")
print()
print("  c) NO INTERFERENCE (SUTVA)")
print("     We assume: Seeing the treatment page does not affect other users")
print("     (Valid in e-commerce without network effects)")
print()
print("CONCLUSION: Causality assumptions APPEAR VALID")
print()
print("="*70)
print()

print("2. IS THE EFFECT CONSISTENT?")
print("-" * 70)
print()
print("Observed heterogeneity:")
print(f"  - NIGHT: {cate_night*100:+.2f}% (worsens)")
print(f"  - DAY: {cate_day*100:+.2f}% (practically neutral)")
print()
if abs(cate_night - cate_day) > 0.001:
    print(f"  NOTABLE HETEROGENEITY: {abs(cate_night - cate_day)*100:.2f}% points")
    print("     -> Effect is NOT consistent")
    print("     -> Recommendation: Do not launch globally")
else:
    print("  Effect is consistent across segments")
print()
print("="*70)
print()

print("3. SUMMARY OF CAUSAL EFFECT")
print("-" * 70)
print()
print(f"ESTIMATED ATE (controlling for confounders): {ate_with_covars*100:+.3f}% points")
print(f"  95% CI: [{ate_covars_ci[0]*100:.3f}%, {ate_covars_ci[1]*100:.3f}%]")
print()
print(f"CATE (NIGHT): {cate_night*100:+.3f}% points")
print(f"CATE (DAY): {cate_day*100:+.3f}% points")
print()
print("CAUSAL INTERPRETATION:")
print(f"  The new treatment CAUSES a change of ~{ate_with_covars*100:.2f}% in conversion")
print(f"  But the effect VARIES by time of day")
print()
print("="*70)

STEP 4: OLS REGRESSION - ATE WITH INFERENCE

MODEL: Conversion = b0 + b1*Treatment

                            OLS Regression Results                            
Dep. Variable:             conversion   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                  0.000
Method:                 Least Squares   F-statistic:                     1.720
Date:                Thu, 09 Apr 2026   Prob (F-statistic):              0.190
Time:                        11:42:04   Log-Likelihood:                -85267.
No. Observations:              290585   AIC:                         1.705e+05
Df Residuals:                  290583   BIC:                         1.706e+05
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------

## STEP 8: Final Project Conclusion

### Synthesis: Correlation -> Causality

**STEP 1 (Traditional A/B Test):**
- Observed: Treatment converts -0.16% less
- Test: P-value = 0.19 (NOT significant)
- Decision: DO NOT LAUNCH

**STEPS 2-3 (Causal Inference):**
- Controlled for confounders (time of day)
- Estimated causal ATE = -0.16% (similar)
- But identified HETEROGENEITY:
  - At night: -0.38% (genuinely worse)
  - During the day: +0.03% (practically neutral)

### Final Answer

#### "Should we launch the new page and for whom?"

1. **Launch globally?** 
   - **NO**
   - Average effect is negative (-0.16%)
   - Also statistically insignificant (p=0.19)

2. **Selective launch (daytime only)?**
   - **MAYBE**
   - During the day the effect is ~neutral (+0.03%)
   - But it's so small it probably isn't worth it
   - Development cost >> potential gain

3. **Launch for NIGHTTIME?**
   - **DEFINITELY NO**
   - At night it is -0.38% (negative)
   - We would harm users in that segment

### Final Recommendation

**DO NOT LAUNCH the new page in its current form**

**Next steps:**
1. Investigate why the new page worsens the result
2. Iterate: improve the design
3. Re-test with improved version
4. If a future test is positive: consider a segmented launch

### Confidence in the Result

- **Methodology:** Rigorous (RCT + Causal Inference)
- **Assumptions:** Valid (Randomization validated)
- **Heterogeneity:** Identified (effect varies by segment)
- **Uncertainty:** Quantified (CI + p-values)
- **Conclusion:** Robust (consistent across methods)

In [8]:
print("="*70)
print("STEP 8: FINAL PROJECT CONCLUSION")
print("="*70)
print()

print("CENTRAL QUESTION:")
print("   Does the treatment really cause the change, and for whom?")
print()
print("="*70)
print()

print("SUMMARY OF FINDINGS")
print("-" * 70)
print()
print("TRADITIONAL A/B TEST (STEPS 1-6):")
print(f"  - Global effect: -0.16%")
print(f"  - P-value: 0.19 (NOT significant)")
print(f"  - Recommendation: DO NOT LAUNCH")
print()
print("CAUSAL INFERENCE (STEPS 7-8):")
print(f"  - ATE (controlling for confounders): -0.16% (similar)")
print(f"  - CATE at night: -0.38%  (worse)")
print(f"  - CATE during day: +0.03% (neutral)")
print(f"  - Heterogeneity: SIGNIFICANT (effect varies)")
print()
print("CAUSALITY ASSUMPTIONS: VALID")
print(f"  Balanced randomization")
print(f"  Full overlap")
print(f"  No interference is assumable")
print()
print("="*70)
print()

print("FINAL DECISION")
print("-" * 70)
print()
print("DO NOT LAUNCH")
print()
print("Reasons:")
print("  1. Global effect is NEGATIVE (-0.16%)")
print("  2. HETEROGENEOUS effect: harms at night, neutral during day")
print("  3. Effect is NOT significant (p=0.19)")
print("  4. Uncertainty is large (CI includes 0)")
print()
print("="*70)
print()

print("NEXT STEPS")
print("-" * 70)
print()
print("1. INVESTIGATE: Why does the new page worsen results?")
print("   - Usability issue?")
print("   - Design problem?")
print("   - Performance issue?")
print()
print("2. ITERATE: Improve the design")
print("   - Fix identified issues")
print("   - Make incremental changes")
print()
print("3. RE-TEST: New A/B experiment")
print("   - Validate whether improvement is real")
print("   - Estimate new causal effect")
print()
print("4. IF FUTURE TEST IS POSITIVE:")
print("   - Consider SELECTIVE launch (daytime only)")
print("   - Continue optimizing for nighttime")
print()
print("="*70)
print()

print("CONFIDENCE IN RESULT")
print("-" * 70)
print()
print("Scale of 1-10:")
print()
print(f"  Methodological rigor: 9/10")
print(f"    -> RCT + Causal Inference framework")
print()
print(f"  Valid assumptions: 9/10")
print(f"    -> Verified randomization, full overlap")
print()
print(f"  Identified heterogeneity: 8/10")
print(f"    -> Clear patterns but still a small effect")
print()
print(f"  Robust conclusion: 8/10")
print(f"    -> Consistent between A/B test and causal inference")
print()
print(f"\n  OVERALL CONFIDENCE: 8.5/10")
print(f"  -> Result is RELIABLE but effect is SMALL")
print()
print("="*70)

STEP 8: FINAL PROJECT CONCLUSION

CENTRAL QUESTION:
   Does the treatment really cause the change, and for whom?


SUMMARY OF FINDINGS
----------------------------------------------------------------------

TRADITIONAL A/B TEST (STEPS 1-6):
  - Global effect: -0.16%
  - P-value: 0.19 (NOT significant)
  - Recommendation: DO NOT LAUNCH

CAUSAL INFERENCE (STEPS 7-8):
  - ATE (controlling for confounders): -0.16% (similar)
  - CATE at night: -0.38%  (worse)
  - CATE during day: +0.03% (neutral)
  - Heterogeneity: SIGNIFICANT (effect varies)

CAUSALITY ASSUMPTIONS: VALID
  Balanced randomization
  Full overlap
  No interference is assumable


FINAL DECISION
----------------------------------------------------------------------

DO NOT LAUNCH

Reasons:
  1. Global effect is NEGATIVE (-0.16%)
  2. HETEROGENEOUS effect: harms at night, neutral during day
  3. Effect is NOT significant (p=0.19)
  4. Uncertainty is large (CI includes 0)


NEXT STEPS
---------------------------------------------

## Summary of Concepts

### What We Learned

| Concept | Lesson |
|----------|----------|
| **Correlation vs Causality** | Seeing a difference does NOT prove causation; we need causal estimation |
| **Confounding** | Omitted variables can bias ATE; controlling for them is critical |
| **ATE vs CATE** | ATE is the average; CATE shows how the effect varies by group |
| **Heterogeneity** | Effects can be opposite in different segments |
| **Regression = Tool** | OLS does not give causality but gives ATE when assumptions are valid |
| **Assumptions Matter** | RCT + Randomization -> we can infer causality |
| **Data-Driven Decision** | Combines statistical significance + effect size + business context |

### The Complete Analysis Workflow

```
Question -> Data -> A/B Test -> Causal Inference -> CATE -> Decision
```

Each step builds on the previous one for a robust conclusion.